# From questionnaire answers to destination recommendations

This is the **minimal operational notebook** for the current prototype. It takes one traveler's profile answers and current-trip answers, compares them with destination attributes, applies the maximum-budget constraint, and returns ranked destinations.

It intentionally excludes sensitivity analysis, fictional test-user comparisons, WAM/WGM evaluation, charts, and model-development discussion.

**Before running:** keep `european_city_destination_dataset_prototype.csv` in the same folder as this notebook. The destination values are synthetic placeholders, so the recommendations are for testing the workflow, not real travel advice.

## 1. Load the destination dataset

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_FILE = Path("european_city_destination_dataset_prototype.csv")

if not DATA_FILE.exists():
    # Google Colab upload option
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
        if not csv_files:
            raise FileNotFoundError("Please upload the destination CSV file.")
        DATA_FILE = Path(csv_files[0])
    except ImportError:
        raise FileNotFoundError(
            "Put european_city_destination_dataset_prototype.csv in the same folder as this notebook."
        )

destinations = pd.read_csv(DATA_FILE)
print(f"Loaded {len(destinations)} destinations.")
display(destinations[["city", "country", "daily_cost_eur_estimate"]].head())

Loaded 50 destinations.


,city,country,daily_cost_eur_estimate
0,Vienna,Austria,165
1,Salzburg,Austria,155
2,Brussels,Belgium,155
3,Bruges,Belgium,145
4,Dubrovnik,Croatia,175


## 2. Enter the questionnaire answers

Replace the example values below with the answers from one participant.

- Numeric preference answers are normalized to **0–1**.
- `authenticity` and `climate` use the exact category labels shown.
- `landscapes` is a list; leave it empty if the user has no preference.
- Budget is in EUR; duration is in days; travelers is the number of people.
- The maximum budget is treated as a **hard constraint**.

In [2]:
user = {
    "profile": {
        "food": 0.70,                       # 0 = not interested, 1 = very interested
        "authenticity": "Everyday local life",  # "Everyday local life", "Tourist-oriented experiences", "Combination of both"
        "relaxation": 0.50,                 # proportion 0–1
        "unique": 0.70,                     # 0–1
        "landscape_importance": 0.70,       # 0–1
        "travel_pace": 0.50,                # 0 = very relaxed, 1 = very active
        "accommodation": 0.70,              # normalized comfort preference 0–1
        "physical_activity": 0.50,          # 0–1
        "price_sensitivity": 0.50,          # retained from questionnaire; not directly used in this prototype
        "safety_importance": 0.80,          # 0–1
        "eco_importance": 0.50              # 0–1
    },
    "trip": {
        "relaxation": 0.60,                 # proportion 0–1
        "unique": 0.70,                     # 0–1
        "landscapes": ["Mountains", "Lakes"],
        "travel_pace": 0.40,                # 0 = very relaxed, 1 = very active
        "accommodation": 0.70,              # normalized 0–1
        "physical_activity": 0.50,          # 0–1
        "climate": "Mild",                  # "Cold", "Cool", "Mild", "Warm", "Hot", "No preference"
        "preferred_budget": 2000,           # EUR; recorded but not used as a constraint in this prototype
        "maximum_budget": 3000,             # EUR; strict upper limit
        "duration_days": 7,
        "travellers": 2
    }
}

# Basic input checks
required_trip = ["maximum_budget", "duration_days", "travellers"]
missing = [key for key in required_trip if key not in user["trip"]]
if missing:
    raise ValueError(f"Missing required trip answers: {missing}")
if user["trip"]["duration_days"] <= 0 or user["trip"]["travellers"] <= 0:
    raise ValueError("Duration and number of travelers must be greater than zero.")

print("Questionnaire answers entered.")

Questionnaire answers entered.


## 3. Define how questionnaire answers match destination attributes

In [3]:
# These are the prototype's current mappings and provisional criterion weights.
NUMERIC_CRITERIA = {
    "food": ("gastronomy", "profile"),
    "authenticity": ("authenticity", "profile"),
    "relaxation": ("relaxation_opportunities", "both"),
    "unique": ("unique_experiences", "both"),
    "travel_pace": ("travel_pace_variety", "both"),
    "accommodation": ("accommodation_comfort_range", "both"),
    "physical_activity": ("adventure_sports_opportunities", "both"),
    "safety": ("safety", "profile"),
    "eco": ("eco_friendliness", "profile"),
}

CRITERION_WEIGHTS = {
    "food": 1.0, "authenticity": 1.0, "relaxation": 1.5,
    "unique": 1.0, "landscape": 1.5, "travel_pace": 1.0,
    "accommodation": 1.5, "physical_activity": 1.0,
    "safety": 1.5, "climate": 1.5, "eco": 0.7
}

## 4. Fuzzy membership and compatibility functions

In [4]:
LINGUISTIC_LABELS = ["Very Low", "Low", "Medium", "High", "Very High"]
CENTRES = np.array([0.0, 0.25, 0.50, 0.75, 1.0])

def five_set_memberships(x):
    """Return memberships in five overlapping fuzzy sets for x in [0, 1]."""
    x = float(np.clip(x, 0.0, 1.0))
    memberships = np.zeros(5, dtype=float)
    memberships[0] = max(0.0, (0.25 - x) / 0.25) if x <= 0.25 else 0.0
    for j in (1, 2, 3):
        left, centre, right = CENTRES[j-1], CENTRES[j], CENTRES[j+1]
        memberships[j] = max(0.0, min((x-left)/(centre-left), (right-x)/(right-centre)))
    memberships[4] = max(0.0, (x - 0.75) / 0.25) if x >= 0.75 else 0.0
    total = memberships.sum()
    return memberships / total if total else memberships

def fuzzy_similarity(user_value, destination_value):
    """Similarity from 0 (very different fuzzy sets) to 1 (same memberships)."""
    mu_user = five_set_memberships(user_value)
    mu_destination = five_set_memberships(destination_value)
    return float(1.0 - 0.5 * np.abs(mu_user - mu_destination).sum())

CLIMATE_ORDER = ["Cold", "Cool", "Mild", "Warm", "Hot"]

def climate_compatibility(user_climate, destination_row):
    if user_climate is None or pd.isna(user_climate) or user_climate == "No preference":
        return 1.0
    destination_climates = [
        climate for climate in CLIMATE_ORDER
        if int(destination_row.get(f"climate_{climate.lower()}", 0)) == 1
    ]
    if not destination_climates:
        return np.nan
    distance = min(abs(CLIMATE_ORDER.index(user_climate) - CLIMATE_ORDER.index(c))
                   for c in destination_climates)
    return {0: 1.0, 1: 0.5}.get(distance, 0.0)

def landscape_compatibility(user_landscapes, destination_row):
    if not user_landscapes:
        return 1.0
    matches = sum(
        int(destination_row.get(f"landscape_{landscape.lower()}", 0)) == 1
        for landscape in user_landscapes
    )
    return matches / len(user_landscapes)

## 5. Calculate compatibility between the traveler and each destination

In [5]:
def numeric_user_value(user_answers, topic):
    profile = user_answers["profile"]
    trip = user_answers["trip"]

    if topic == "food":
        return profile.get("food")
    if topic == "authenticity":
        mapping = {
            "Tourist-oriented experiences": 0.20,
            "Combination of both": 0.50,
            "Everyday local life": 0.90
        }
        return mapping.get(profile.get("authenticity"))
    if topic == "relaxation":
        return profile.get("relaxation"), trip.get("relaxation")
    if topic == "unique":
        return profile.get("unique"), trip.get("unique")
    if topic == "travel_pace":
        return profile.get("travel_pace"), trip.get("travel_pace")
    if topic == "accommodation":
        return profile.get("accommodation"), trip.get("accommodation")
    if topic == "physical_activity":
        return profile.get("physical_activity"), trip.get("physical_activity")
    if topic == "eco":
        return profile.get("eco_importance")
    return None

def destination_numeric_score(destination_row, topic):
    column = NUMERIC_CRITERIA[topic][0]
    return float(destination_row[column])

def get_criterion_scores(user_answers, destination_row):
    profile = user_answers["profile"]
    trip = user_answers["trip"]
    scores = {}

    # Profile-only criteria
    for topic in ["food", "authenticity", "eco"]:
        user_value = numeric_user_value(user_answers, topic)
        if user_value is not None:
            scores[topic] = fuzzy_similarity(
                user_value, destination_numeric_score(destination_row, topic)
            )

    # Shared profile/trip criteria: 30% profile + 70% current trip
    for topic in ["relaxation", "unique", "travel_pace", "accommodation", "physical_activity"]:
        profile_value, trip_value = numeric_user_value(user_answers, topic)
        destination_value = destination_numeric_score(destination_row, topic)
        profile_match = fuzzy_similarity(profile_value, destination_value)
        trip_match = fuzzy_similarity(trip_value, destination_value)
        scores[topic] = 0.30 * profile_match + 0.70 * trip_match

    scores["landscape"] = landscape_compatibility(trip.get("landscapes", []), destination_row)
    scores["climate"] = climate_compatibility(trip.get("climate"), destination_row)

    # Current prototype assumption: higher destination safety is preferred;
    # the user's safety importance scales the criterion weight below.
    scores["safety"] = float(destination_row["safety"])
    return scores

def get_user_criterion_weights(user_answers):
    weights = CRITERION_WEIGHTS.copy()
    weights["landscape"] *= max(0.05, user_answers["profile"].get("landscape_importance", 0.5))
    weights["safety"] *= max(0.05, user_answers["profile"].get("safety_importance", 0.5))
    return weights

## 6. Aggregate scores, apply the budget constraint, and rank destinations

In [6]:
def aggregate_wam(score_values, weights):
    values = np.asarray(score_values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        return np.nan
    return float(np.average(values[valid], weights=weights[valid]))

def aggregate_wgm(score_values, weights, epsilon=1e-9):
    values = np.asarray(score_values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        return np.nan
    values = np.clip(values[valid], epsilon, 1.0)
    normalized_weights = weights[valid] / weights[valid].sum()
    return float(np.exp(np.sum(normalized_weights * np.log(values))))

def recommend(user_answers, destination_df):
    weights = get_user_criterion_weights(user_answers)
    criterion_names = list(weights)

    records = []
    for _, destination in destination_df.iterrows():
        criterion_scores = get_criterion_scores(user_answers, destination)
        records.append({
            "destination_id": destination["destination_id"],
            "city": destination["city"],
            "country": destination["country"],
            **criterion_scores
        })

    scores = pd.DataFrame(records)

    costs = destination_df[["destination_id", "daily_cost_eur_estimate"]].copy()
    costs["estimated_trip_cost_eur"] = (
        costs["daily_cost_eur_estimate"]
        * user_answers["trip"]["duration_days"]
        * user_answers["trip"]["travellers"]
    )
    costs["eligible"] = costs["estimated_trip_cost_eur"] <= user_answers["trip"]["maximum_budget"]

    result = scores.merge(
        costs[["destination_id", "estimated_trip_cost_eur", "eligible"]],
        on="destination_id", how="left"
    )

    weight_values = [weights[name] for name in criterion_names]
    result["WAM_score"] = result.apply(
        lambda row: aggregate_wam([row[name] for name in criterion_names], weight_values),
        axis=1
    )
    result["WGM_score"] = result.apply(
        lambda row: aggregate_wgm([row[name] for name in criterion_names], weight_values),
        axis=1
    )

    # Strictly exclude destinations over the maximum budget
    eligible = result[result["eligible"]].copy()
    if eligible.empty:
        return result, eligible, eligible

    eligible["WAM_rank"] = eligible["WAM_score"].rank(
        ascending=False, method="min"
    ).astype(int)
    eligible["WGM_rank"] = eligible["WGM_score"].rank(
        ascending=False, method="min"
    ).astype(int)

    wam_ranked = eligible.sort_values("WAM_score", ascending=False)
    wgm_ranked = eligible.sort_values("WGM_score", ascending=False)
    return result, wam_ranked, wgm_ranked

## 7. Get the recommendations

In [7]:
all_scores, wam_recommendations, wgm_recommendations = recommend(user, destinations)

if wam_recommendations.empty:
    print("No destinations meet the maximum budget. Consider changing the budget or trip details.")
else:
    print("Top recommendations — Weighted Arithmetic Mean (WAM)")
    display(wam_recommendations[
        ["city", "country", "estimated_trip_cost_eur", "WAM_score", "WAM_rank"]
    ].head(5).round(3))

    print("Top recommendations — Weighted Geometric Mean (WGM)")
    display(wgm_recommendations[
        ["city", "country", "estimated_trip_cost_eur", "WGM_score", "WGM_rank"]
    ].head(5).round(3))

Top recommendations — Weighted Arithmetic Mean (WAM)


,city,country,estimated_trip_cost_eur,WAM_score,WAM_rank
28,Amsterdam,Netherlands,2450,0.700,1
45,Lucerne,Switzerland,2870,0.694,2
7,Brno,Czechia,1904,0.670,3
6,Prague,Czechia,1610,0.641,4
29,Rotterdam,Netherlands,2940,0.637,5


Top recommendations — Weighted Geometric Mean (WGM)


,city,country,estimated_trip_cost_eur,WGM_score,WGM_rank
28,Amsterdam,Netherlands,2450,0.670,1
17,Munich,Germany,1918,0.597,2
12,Helsinki,Finland,2226,0.583,3
49,Edinburgh,United Kingdom,2170,0.544,4
38,Ljubljana,Slovenia,1274,0.509,5


## Notes about this prototype

- The notebook implements the current prototype's mappings, 30% profile / 70% trip blending for shared criteria, provisional weights, and strict maximum-budget exclusion.
- It returns rankings using both WAM and WGM so the recommendation output can be inspected.
- Some questionnaire answers are not yet used in the scoring pipeline (for example, preferred budget and price sensitivity); maximum budget is used as a hard constraint.
- The destination dataset currently contains synthetic placeholder values. Do not interpret the rankings as validated real-world recommendations.